## This demo showcases the implementation of user story 589

This notebook shows the ability to stage a STAC ItemCollection from a single link

In [1]:
import requests
import os
import pprint
import time
import pystac
# Init environment before running a demo notebook.
from resources.utils import *

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)
session = requests.Session()
auxip_client, cadip_client, catalog_client, staging_client = init_demo()

if os.getenv("RSPY_LOCAL_MODE") == "1":
    href_cadip = "http://rs-server-cadip:8000"
    href_adgs = "http://rs-server-adgs:8000"
else:
    href_cadip = href_adgs = os.environ["RSPY_WEBSITE"]
    session.cookies.set("session", os.environ["RSPY_OAUTH2_COOKIE"])

cadip_collection_id = "cadip_sentinel1"
adgs_collection_id = "adgs"
TIMEOUT = 10

DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.1"


Auxip service: http://rs-server-adgs:8000/auxip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000


In [2]:
# Init the dask cluster
from resources.dask_utils import *
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.dask_utils import *

# Use the staging cluster
dask_gateway = dask_gateway_staging
dask_cluster = dask_cluster_staging

Connecting to dask gateway for 'dask-staging': http://dask-staging:8000 ...
Create new dask cluster
Dask dashboard for 'dask-staging': http://localhost:8701/clusters/9f6936b9fddc4a2e84591fcb968fb865/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-staging' are up: 0/2
Dask workers for 'dask-staging' are up: 2/2


In [3]:
# Create a test collection 
collection = create_test_collection()

### Check the newly created collection with the rs-server-catalog. It should be empty.

In [4]:
# Check the catalog for my_test_collection
collection = catalog_client.get_collection(TEST_COLLECTION)
for item in collection.get_items():
    print(f"Item {item.id} has {len(item.assets)} assets")

### Stage one item from Cadip station and one item from adgs station using links

In [5]:
staging_resp_list = []
staging_info = {
    "station": ["cadip", "auxip"],
    "id": ["S1A_20231120061537234567", "S1A_OPER_AUX_PREORB_OPOD_20240527T062732_V20240527T062732_20240527T062732.EOF"],
    "collection": ["cadip_sentinel1", "adgs"],
    "href": ["http://rs-server-cadip:8000", "http://rs-server-adgs:8000"]
}

for i in range(len(staging_info["station"])):
    staging_link = f"{staging_info['href'][i]}/{staging_info['station'][i]}/search?ids={staging_info['id'][i]}&collections={staging_info['collection'][i]}"
    staging_resp_list.append(staging_client.run_staging(staging_link, TEST_COLLECTION))

In [6]:
timeout = 120
started_job_id_list = []

for resp in staging_resp_list:
    started_job_id_list.append(resp["jobID"])
    while timeout > 0:
        if "running" not in resp["status"]:
            break
        job_info = staging_client.get_job_info(resp["jobID"])
        pprint.PrettyPrinter(indent=4).pprint(job_info)
        print("\n")
        if "successful" in job_info["status"]:
            print(" ----- Job COMPLETED \n")
            break
        if "failed" in job_info["status"]:
            print("-----Job FAILED \n")
            break
        time.sleep(2)
        timeout -= 2

{   'created': '2025-04-01T22:00:42Z',
    'jobID': '0a57cbb4-5c64-490e-b918-9febe452f25b',
    'message': 'Sending tasks to the dask cluster',
    'processID': 'staging',
    'progress': 0,
    'started': '2025-04-01T22:00:42Z',
    'status': 'running',
    'type': 'process',
    'updated': '2025-04-01T22:00:42Z'}


{   'created': '2025-04-01T22:00:42Z',
    'jobID': '0a57cbb4-5c64-490e-b918-9febe452f25b',
    'message': 'In progress',
    'processID': 'staging',
    'progress': 67,
    'started': '2025-04-01T22:00:42Z',
    'status': 'running',
    'type': 'process',
    'updated': '2025-04-01T22:00:47Z'}


{   'created': '2025-04-01T22:00:42Z',
    'jobID': '0a57cbb4-5c64-490e-b918-9febe452f25b',
    'message': 'Finished',
    'processID': 'staging',
    'progress': 100,
    'started': '2025-04-01T22:00:42Z',
    'status': 'successful',
    'type': 'process',
    'updated': '2025-04-01T22:00:49Z'}


 ----- Job COMPLETED 

{   'created': '2025-04-01T22:00:42Z',
    'jobID': 'c801340a

### Check the catalog for my_test_collection. Ten items should be present now (one from CADIP station and nine from AUXIP station)

In [7]:
# Check the catalog for my_test_collection
result = list(catalog_client.get_collection(TEST_COLLECTION).get_items())
print (f"{len(result)} items before removing")
for item in result:
    print(f"Item {item.id} has {len(item.assets)} assets")
assert len(result) == 2

2 items before removing
Item S1A_OPER_AUX_PREORB_OPOD_20240527T062732_V20240527T062732_20240527T062732.EOF has 1 assets
Item S1A_20231120061537234567 has 60 assets


### Check the jobs table

In [8]:
# Check that each of the job previously launched are successful
for job_id in started_job_id_list:
    job_results = staging_client.get_job_results(job_id)
    print(f"Results from job {job_id}: {job_results}")
    assert job_results == "successful"

Results from job 0a57cbb4-5c64-490e-b918-9febe452f25b: successful
Results from job c801340a-1295-4084-8573-8389c3afb1ec: successful


### Delete the whole collection

In [9]:
result = catalog_client.remove_collection(TEST_COLLECTION)
assert result.json()["deleted collection"] == TEST_COLLECTION
pp.pprint(result.json())

{'deleted collection': 'my_test_collection'}


In [10]:
if local_mode:

    # You can scale the clusters to 0 workers
    dask_gateway.scale_cluster(dask_cluster.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway, dask_cluster.name)

# Close the python objects
close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.

Shutting down cluster '9f6936b9fddc4a2e84591fcb968fb865' ...
